In [95]:
import time
import pandas as pd
from natnet import DataDescriptions, DataFrame, NatNetClient
import matplotlib.pyplot as plt
import numpy as np

In [76]:
num_frames = 0
rigid_bodies = []

def receive_new_frame(data_frame: DataFrame):
    global num_frames
    global rigidbodies
    num_frames += 1
    for rigid_body in data_frame.rigid_bodies:
        if rigid_body.id_num == 6:
            rigid_bodies.append(rigid_body)


def receive_new_desc(desc: DataDescriptions):
    print("Received data descriptions.")


In [79]:
num_frames = 0
if __name__ == "__main__":
    streaming_client = NatNetClient(server_ip_address="192.168.1.92", local_ip_address="0.0.0.0", use_multicast=True)
    streaming_client.on_data_description_received_event.handlers.append(receive_new_desc)
    streaming_client.on_data_frame_received_event.handlers.append(receive_new_frame)

    with streaming_client:
        a = streaming_client.request_modeldef()

        for i in range(10):
            time.sleep(1)
            streaming_client.update_sync()
            print(f"Received {num_frames} frames in {i + 1}s")

    df = pd.DataFrame([rb.pos for rb in rigid_bodies], columns=["x", "y", "z"])
    df.to_csv("rigid_bodies.csv", index=False)



Received data descriptions.
Received 0 frames in 1s
Received 0 frames in 2s
Received 0 frames in 3s
Received 0 frames in 4s
Received 0 frames in 5s
Received 0 frames in 6s
Received 0 frames in 7s
Received 0 frames in 8s
Received 0 frames in 9s
Received 0 frames in 10s


In [98]:
#Scatter plot of the rigid body positions 
df = pd.read_csv("rigid_bodies.csv")
print(df.__len__)
df = df[200:450]

def plot_3d_scatter(df: pd.DataFrame, name: str):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(df['x'], df['y'], df['z'], c='r', marker='o', s=5)
    ax.set_xlabel('X (mm)')
    ax.set_ylabel('Y (mm)')
    ax.set_zlabel('Z (mm)')
    ax.set_title('Rigid Body Positions')
    plt.savefig(name)
    plt.close() 

plot_3d_scatter(df, "rigid_body_positions.png")



<bound method DataFrame.__len__ of             x         y         z
0   -0.602588  1.511815  0.157858
1   -0.602618  1.511866  0.157854
2   -0.602649  1.511909  0.157620
3   -0.602572  1.511802  0.157870
4   -0.602659  1.511846  0.157841
..        ...       ...       ...
510 -0.093541  0.755141  1.396109
511 -0.137798  0.760326  1.454597
512 -0.140128  0.760332  1.457763
513 -0.143203  0.760271  1.461120
514 -0.145661  0.760276  1.464186

[515 rows x 3 columns]>


In [108]:
def interpolate_3d_points(df, target_points=1000, x_col='x', y_col='y', z_col='z'):
    """
    Linearly interpolate 3D points along a curve using arc-length parameterization.
    
    Args:
        df: DataFrame with columns for x, y, z coordinates
        target_points: number of output points (default 1000)
        x_col, y_col, z_col: column names in the DataFrame
    
    Returns:
        DataFrame with interpolated points
    """
    points = df[[x_col, y_col, z_col]].values

    # Compute cumulative arc length as the parameter
    deltas = np.diff(points, axis=0)
    segment_lengths = np.linalg.norm(deltas, axis=1)
    arc_length = np.concatenate([[0], np.cumsum(segment_lengths)])
    arc_length_normalized = arc_length / arc_length[-1]  # normalize to [0, 1]

    # New parameter values evenly spaced along the curve
    t_new = np.linspace(0, 1, target_points)

    # Interpolate each dimension independently
    x_new = np.interp(t_new, arc_length_normalized, points[:, 0])
    y_new = np.interp(t_new, arc_length_normalized, points[:, 1])
    z_new = np.interp(t_new, arc_length_normalized, points[:, 2])

    return pd.DataFrame({x_col: x_new, y_col: y_new, z_col: z_new})


df = pd.read_csv("rigid_bodies.csv")[200:450]
interpolated_df = interpolate_3d_points(df, target_points=1000)
interpolated_df.to_csv("interpolated_rigid_bodies.csv", index=False)

plot_3d_scatter(interpolated_df, "interpolated_rigid_body_positions.png")